# LeetCode 954 — Array of Doubled Pairs bằng Python

Link đề: https://leetcode.com/problems/array-of-doubled-pairs/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Hash Table (bảng băm), Greedy (tham lam), Sorting (sắp xếp)

Skeleton LeetCode:

```python
class Solution:
    def canReorderDoubled(self, arr: List[int]) -> bool:
        ...
```

Ghi chú: ta phải chia mảng thành **các cặp `(x, 2x)`**: mỗi phần tử thuộc đúng một cặp, trong cặp phần tử thứ hai gấp đôi phần tử thứ nhất. Chìa khóa là **thứ tự xử lý**: duyệt các giá trị theo **giá trị tuyệt đối tăng dần**; phần tử có giá trị tuyệt đối nhỏ nhất `x` chỉ có thể là phần tử **đầu** của cặp (vì `x / 2` có giá trị tuyệt đối nhỏ hơn, mà không còn), nên nó **bắt buộc** ghép với một `2x`. Cạm bẫy: số âm (thứ tự `|x|`) và số `0` (`0 = 2 * 0`, ghép với chính nó).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có sắp xếp lại được mảng độ dài chẵn để `arr[2i + 1] = 2 * arr[2i]` với mọi `i` hay không.
2. Nắm các công cụ Python cần thiết: `collections.Counter`, `sorted(..., key=abs)`, cập nhật bộ đếm `cnt[2 * x] -= cnt[x]`, và tách phần lẻ `x = 2^k * odd`.
3. Hiểu nguyên lý tham lam: xử lý theo `|x|` tăng dần, mỗi `x` còn dư phải ghép với `2x`; nếu không đủ `2x` thì trả `False`.
4. Đi qua 4 cách: quay lui thử mọi cách ghép cặp (brute, oracle), tham lam theo `|x|` với `Counter` (cách nộp chính), tham lam tách số âm và số dương, và phân rã theo **chuỗi gấp đôi** của phần lẻ; đối chiếu bằng oracle và kiểm tra đáp án có thể dựng lại.
5. Tránh các lỗi kinh điển: sắp xếp theo giá trị (không phải `|x|`), quên trừ số đã ghép, chỉ kiểm tra "có mặt" thay vì đếm số lượng, và duyệt `Counter` theo thứ tự chèn.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `arr` có độ dài **chẵn**. Trả về `true` nếu có thể **sắp xếp lại** `arr` sao cho `arr[2 * i + 1] = 2 * arr[2 * i]` với mọi `0 <= i < len(arr) / 2`, ngược lại trả `false`.

**Ví dụ 1:** `arr = [3,1,3,6]` -> `false`.

**Ví dụ 2:** `arr = [2,1,2,6]` -> `false`.

**Ví dụ 3:** `arr = [4,-2,2,-4]` -> `true` (hai nhóm `[-2,-4]` và `[2,4]`, sắp xếp thành `[-2,-4,2,4]`).

**Ràng buộc:**

- `2 <= arr.length <= 3 * 10^4`
- `arr.length` là số chẵn.
- `-10^5 <= arr[i] <= 10^5`

In [1]:
from typing import List
from collections import Counter
import random, time

examples = [
    ([3, 1, 3, 6], False),
    ([2, 1, 2, 6], False),
    ([4, -2, 2, -4], True),
]
for arr, expected in examples:
    print(f"arr={arr} kỳ vọng={expected}")

arr=[3, 1, 3, 6] kỳ vọng=False
arr=[2, 1, 2, 6] kỳ vọng=False
arr=[4, -2, 2, -4] kỳ vọng=True


## 3. Kiến thức nền

### 3.1. Đếm bằng `Counter`

`Counter(arr)` đếm số lần xuất hiện của mỗi giá trị. Bài toán chỉ phụ thuộc vào **đa tập** (thứ tự ban đầu không quan trọng vì ta được phép sắp xếp lại).

In [2]:
cnt = Counter([4, -2, 2, -4])
print(cnt, "| cnt[2] =", cnt[2], "| cnt[100] =", cnt[100], "(giá trị vắng mặt cho 0)")

Counter({4: 1, -2: 1, 2: 1, -4: 1}) | cnt[2] = 1 | cnt[100] = 0 (giá trị vắng mặt cho 0)


### 3.2. Sắp xếp theo giá trị tuyệt đối

`sorted(cnt, key=abs)` duyệt các **giá trị khác nhau** theo `|x|` tăng dần: `0, 1, -1, 2, -2, ...` (cùng `|x|` thì giữ thứ tự gặp). Với số âm, "gấp đôi" làm `|x|` **tăng**: `-2 -> -4`. Vì vậy trục "nhỏ tới lớn" đúng là `|x|`.

In [3]:
print(sorted(Counter([4, -2, 2, -4, 0, 1, -1]), key=abs))
print("sắp xếp theo giá trị (sai trục):", sorted([4, -2, 2, -4]))

[0, 1, -1, -2, 2, 4, -4]
sắp xếp theo giá trị (sai trục): [-4, -2, 2, 4]


### 3.3. Số `0` và quy tắc "đầu cặp"

`0 = 2 * 0`, nên `0` chỉ ghép được với `0`: số lượng `0` phải **chẵn**. Vì độ dài mảng chẵn, số lượng `0` lẻ kéo theo số lượng phần tử khác `0` lẻ nên chắc chắn không ghép hết; ta vẫn kiểm tra riêng `cnt[0] % 2` cho rõ ràng và bỏ qua `x = 0` trong vòng lặp tham lam (để `x = 0` không tự trừ chính nó một cách khó hiểu).

## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 3: `[4, -2, 2, -4]`: đếm `{4:1, -2:1, 2:1, -4:1}`. Theo `|x|` tăng: `-2, 2, 4, -4`.

- `x = -2` (còn 1): cần `-4` (còn 1): ghép, `cnt[-2] = 0`, `cnt[-4] = 0`.
- `x = 2` (còn 1): cần `4` (còn 1): ghép.
- `x = 4`, `x = -4`: đã hết.

Thành công: `true`.

Ví dụ 1: `[3, 1, 3, 6]`: theo `|x|`: `1, 3, 6`. `x = 1` (còn 1) cần `2`: không có ⇒ `false`.

## 5. Cách 1 — Brute Force: thử mọi cách ghép cặp

**Ý tưởng.** Quay lui: lấy một phần tử còn lại `e` (phần tử đầu danh sách); nó là **đầu cặp** (ghép với một `2e`) hoặc **cuối cặp** (ghép với một `e / 2`, khi `e` chẵn). Thử cả hai khả năng với mọi phần tử đối tác còn lại; thành công khi hết phần tử. Đây là định nghĩa trực tiếp nên cũng là **oracle**.

In [4]:
def reorder_brute(arr: List[int]) -> bool:
    items = list(arr)

    def go(rest: List[int]) -> bool:
        if not rest:
            return True
        e = rest[0]
        others = rest[1:]
        for idx, y in enumerate(others):
            if y == 2 * e or (e % 2 == 0 and y == e // 2):        # e đứng trước 2e, hoặc e = 2y
                if go(others[:idx] + others[idx + 1:]):
                    return True
        return False

    return go(items)

for arr, expected in examples:
    got = reorder_brute(arr)
    print(arr, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[3, 1, 3, 6] -> False OK
[2, 1, 2, 6] -> False OK
[4, -2, 2, -4] -> True OK


**Độ phức tạp và nút thắt.** Có thể thử rất nhiều cách ghép (mũ). Xem thời gian với `n` chẵn tăng:

In [5]:
rng = random.Random(954)
for n in (8, 10, 12, 14):
    base = [rng.randint(1, 3) for _ in range(n // 2)]
    arr = base + [2 * x for x in base]                  # chắc chắn True: nhưng cần thử nhiều nhánh
    rng.shuffle(arr)
    t0 = time.perf_counter()
    r = reorder_brute(arr)
    print(f"n={n:2}: {1000 * (time.perf_counter() - t0):9.1f} ms (kết quả {r})")

n= 8:       0.0 ms (kết quả True)
n=10:       0.0 ms (kết quả True)
n=12:       0.0 ms (kết quả True)
n=14:       0.0 ms (kết quả True)


Nút thắt: ta thử nhiều cách ghép trong khi cách ghép của phần tử **nhỏ nhất theo `|x|`** bị **ép buộc**.

## 6. Ý tưởng thuật toán / bước đột phá

**Phần tử có `|x|` nhỏ nhất bắt buộc là đầu cặp.** Xét giá trị `x` có `|x|` nhỏ nhất còn lại (và `x != 0`). Nếu `x` là phần tử **thứ hai** của một cặp thì cặp đó là `(x / 2, x)` với `|x / 2| < |x|`: nhưng không còn phần tử nào nhỏ hơn `|x|` (bị ghép xong hoặc không có), mâu thuẫn. Vậy mọi `x` còn lại đều là đầu cặp, và cần đúng một `2x` cho mỗi bản sao; ghép `cnt[x]` bản `x` với `cnt[x]` bản `2x`. Lặp theo `|x|` tăng dần.

| Cách | Ý tưởng |
|---|---|
| 2 | `Counter` + `sorted(..., key=abs)`, ghép `x` với `2x` (nộp) |
| 3 | Tách số **âm** và **dương**: xử lý riêng, không dùng `key=abs` |
| 4 | Phân rã theo **chuỗi gấp đôi**: nhóm theo phần lẻ, trong mỗi chuỗi ghép liên tiếp |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — `Counter` và duyệt theo `|x|` (cách nộp chính)

**Giải thích từng đoạn.**

- `cnt = Counter(arr)`.
- `for x in sorted(cnt, key=abs)`: duyệt giá trị khác nhau theo `|x|` tăng dần.
- Nếu `cnt[2 * x] < cnt[x]` thì trả `False`; ngược lại `cnt[2 * x] -= cnt[x]`.
- **Số `0`:** `2 * 0 = 0` nên phép `cnt[2 * x] -= cnt[x]` với `x = 0` tự trừ chính nó (vô nghĩa). Ta kiểm tra riêng `cnt[0]` chẵn rồi bỏ qua `x = 0`.

In [6]:
def reorder_counter(arr: List[int]) -> bool:
    cnt = Counter(arr)
    if cnt[0] % 2:
        return False
    for x in sorted(cnt, key=abs):
        if x == 0:
            continue
        if cnt[2 * x] < cnt[x]:
            return False
        cnt[2 * x] -= cnt[x]
    return True

for arr, expected in examples:
    got = reorder_counter(arr)
    print(arr, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[3, 1, 3, 6] -> False OK
[2, 1, 2, 6] -> False OK
[4, -2, 2, -4] -> True OK


**Code nộp LeetCode (Cách 2):**

In [7]:
class Solution:
    def canReorderDoubled(self, arr: List[int]) -> bool:
        from collections import Counter
        cnt = Counter(arr)
        if cnt[0] % 2:
            return False
        for x in sorted(cnt, key=abs):
            if x == 0:
                continue
            if cnt[2 * x] < cnt[x]:
                return False
            cnt[2 * x] -= cnt[x]
        return True

sol_counter = Solution().canReorderDoubled
assert [sol_counter(list(a)) for a, _ in examples] == [e for _, e in examples]
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** `O(n + d log d)` với `d` là số giá trị khác nhau (sắp xếp các giá trị khác nhau); `O(n)` bộ nhớ.

---

### 7.2. Cách 3 — Tách số âm và số dương

**Ý tưởng khác.** Thay `key=abs` bằng hai lượt: các số **dương** theo giá trị tăng dần (`x` cần `2x`), các số **âm** theo giá trị **giảm dần** (tức `|x|` tăng: `-1, -2, -3, ...`; `x` cần `2x`, ví dụ `-1` cần `-2`). Số `0` kiểm tra riêng.

**Giải thích từng đoạn.** `sorted(positives)` tăng dần; `sorted(negatives, reverse=True)` giảm dần (gần `0` trước). Cập nhật giống Cách 2.

In [8]:
def reorder_split_sign(arr: List[int]) -> bool:
    cnt = Counter(arr)
    if cnt[0] % 2:
        return False
    positives = sorted(x for x in cnt if x > 0)
    negatives = sorted((x for x in cnt if x < 0), reverse=True)
    for group in (positives, negatives):
        for x in group:
            if cnt[2 * x] < cnt[x]:
                return False
            cnt[2 * x] -= cnt[x]
    return True

for arr, expected in examples:
    got = reorder_split_sign(arr)
    print(arr, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[3, 1, 3, 6] -> False OK
[2, 1, 2, 6] -> False OK
[4, -2, 2, -4] -> True OK


**Code nộp LeetCode (Cách 3):**

In [9]:
class Solution:
    def canReorderDoubled(self, arr: List[int]) -> bool:
        from collections import Counter
        cnt = Counter(arr)
        if cnt[0] % 2:
            return False
        for x in sorted(v for v in cnt if v > 0):
            if cnt[2 * x] < cnt[x]:
                return False
            cnt[2 * x] -= cnt[x]
        for x in sorted((v for v in cnt if v < 0), reverse=True):
            if cnt[2 * x] < cnt[x]:
                return False
            cnt[2 * x] -= cnt[x]
        return True

sol_split = Solution().canReorderDoubled
assert [sol_split(list(a)) for a, _ in examples] == [e for _, e in examples]
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** `O(n + d log d)` thời gian, `O(n)` bộ nhớ.

---

### 7.3. Cách 4 — Phân rã theo chuỗi gấp đôi

**Ý tưởng khác.** Mỗi số khác `0` viết duy nhất dưới dạng `x = s * 2^k * o` với `o` lẻ dương và `s = +-1`. Các số cùng `(s, o)` tạo **chuỗi** `o, 2o, 4o, ...` (nhân với `s`). Cặp `(x, 2x)` chỉ nối hai phần tử **liền kề** trong cùng một chuỗi. Trong chuỗi, gọi `c_k` là số lần xuất hiện của phần tử bậc `k`: phải ghép `c_0` với `c_1`, phần dư `c_1 - c_0` với `c_2`, ... Cần mọi hiệu `c_k - r_{k-1} >= 0` và hiệu cuối bằng `0`, với `r_k = c_k - r_{k-1}`.

**Giải thích từng đoạn.** `r` là số bản của bậc `k - 1` còn chưa ghép (phải ghép với bậc `k`): `r = 0` ban đầu; với mỗi bậc: nếu `c_k < r` thì sai; `r = c_k - r`. Cuối chuỗi cần `r == 0`.

In [10]:
def reorder_chains(arr: List[int]) -> bool:
    cnt = Counter(arr)
    if cnt[0] % 2:
        return False
    chains = {}                                            # (dấu, phần lẻ) -> {bậc k: số lần}
    for x, c in cnt.items():
        if x == 0:
            continue
        sign = 1 if x > 0 else -1
        v, k = abs(x), 0
        while v % 2 == 0:
            v //= 2
            k += 1
        chains.setdefault((sign, v), {})[k] = c
    for chain in chains.values():
        r = 0
        for k in range(max(chain) + 1):
            c = chain.get(k, 0)
            if c < r:
                return False
            r = c - r
        if r != 0:
            return False
    return True

for arr, expected in examples:
    got = reorder_chains(arr)
    print(arr, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[3, 1, 3, 6] -> False OK
[2, 1, 2, 6] -> False OK
[4, -2, 2, -4] -> True OK


**Code nộp LeetCode (Cách 4):**

In [11]:
class Solution:
    def canReorderDoubled(self, arr: List[int]) -> bool:
        from collections import Counter
        cnt = Counter(arr)
        if cnt[0] % 2:
            return False
        chains = {}
        for x, c in cnt.items():
            if x == 0:
                continue
            sign = 1 if x > 0 else -1
            v, k = abs(x), 0
            while v % 2 == 0:
                v //= 2
                k += 1
            chains.setdefault((sign, v), {})[k] = c
        for chain in chains.values():
            r = 0
            for k in range(max(chain) + 1):
                c = chain.get(k, 0)
                if c < r:
                    return False
                r = c - r
            if r != 0:
                return False
        return True

sol_chain = Solution().canReorderDoubled
assert [sol_chain(list(a)) for a, _ in examples] == [e for _, e in examples]
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** `O(n + d log V)` thời gian (tách lũy thừa của `2` mỗi giá trị), `O(d)` bộ nhớ phụ.

## 8. TRACE từng bước

### 8.1. Cách 2 cho ví dụ 3: `[4, -2, 2, -4]`

In [12]:
def trace_counter(arr):
    cnt = Counter(arr)
    print("đếm:", dict(cnt), "| số 0 chẵn?", cnt[0] % 2 == 0)
    if cnt[0] % 2:
        return False
    for x in sorted(cnt, key=abs):
        if x == 0:
            continue
        need = cnt[x]
        have = cnt[2 * x]
        if have < need:
            print(f"x={x}: cần {need} bản của {2 * x} nhưng chỉ có {have} -> False")
            return False
        cnt[2 * x] -= need
        print(f"x={x}: ghép {need} cặp ({x}, {2 * x}); còn cnt[{2 * x}] = {cnt[2 * x]}")
    return True

assert trace_counter([4, -2, 2, -4]) is True
print()
assert trace_counter([3, 1, 3, 6]) is False

đếm: {4: 1, -2: 1, 2: 1, -4: 1} | số 0 chẵn? True
x=-2: ghép 1 cặp (-2, -4); còn cnt[-4] = 0
x=2: ghép 1 cặp (2, 4); còn cnt[4] = 0
x=4: ghép 0 cặp (4, 8); còn cnt[8] = 0
x=-4: ghép 0 cặp (-4, -8); còn cnt[-8] = 0

đếm: {3: 2, 1: 1, 6: 1} | số 0 chẵn? True
x=1: cần 1 bản của 2 nhưng chỉ có 0 -> False


### 8.2. Cách 3: thứ tự duyệt `[4, -2, 2, -4]`

Dương tăng dần `[2, 4]`, âm giảm dần `[-2, -4]` (gần `0` trước): `-2` cần `-4`, rồi `-4` không còn dư.

In [13]:
cnt = Counter([4, -2, 2, -4])
print("dương:", sorted(x for x in cnt if x > 0), "| âm:", sorted((x for x in cnt if x < 0), reverse=True))

dương: [2, 4] | âm: [-2, -4]


### 8.3. Cách 4: chuỗi gấp đôi của `[1, 2, 2, 4, 4, 8, 8, 16]`

Tất cả cùng phần lẻ `1`: chuỗi `1, 2, 4, 8, 16` với số lần `c = [1, 2, 2, 2, 1]`. `r`: `0 -> c0 - 0 = 1 -> c1 - 1 = 1 -> c2 - 1 = 1 -> c3 - 1 = 1 -> c4 - 1 = 0`: thành công.

In [14]:
c = [1, 2, 2, 2, 1]
r = 0
for k, ck in enumerate(c):
    assert ck >= r
    r = ck - r
    print(f"bậc {k}: c={ck}, r (cần ghép với bậc kế) = {r}")
assert r == 0 and reorder_chains([1, 2, 2, 4, 4, 8, 8, 16]) is True

bậc 0: c=1, r (cần ghép với bậc kế) = 1
bậc 1: c=2, r (cần ghép với bậc kế) = 1
bậc 2: c=2, r (cần ghép với bậc kế) = 1
bậc 3: c=2, r (cần ghép với bậc kế) = 1
bậc 4: c=1, r (cần ghép với bậc kế) = 0


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (`Counter` + duyệt theo `|x|`): ngắn, `O(n + d log d)`.

```python
class Solution:
    def canReorderDoubled(self, arr: List[int]) -> bool:
        cnt = Counter(arr)
        if cnt[0] % 2:
            return False
        for x in sorted(cnt, key=abs):
            if x == 0:
                continue
            if cnt[2 * x] < cnt[x]:
                return False
            cnt[2 * x] -= cnt[x]
        return True
```

In [15]:
class Solution:
    def canReorderDoubled(self, arr: List[int]) -> bool:
        cnt = Counter(arr)
        if cnt[0] % 2:
            return False
        for x in sorted(cnt, key=abs):
            if x == 0:
                continue
            if cnt[2 * x] < cnt[x]:
                return False
            cnt[2 * x] -= cnt[x]
        return True

sol = Solution()
for arr, expected in examples:
    got = sol.canReorderDoubled(list(arr))
    print(arr, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[3, 1, 3, 6] -> False OK
[2, 1, 2, 6] -> False OK
[4, -2, 2, -4] -> True OK


## 10. Vì sao đúng

**(a) Đa tập và cặp.** Sắp xếp lại được ⇔ đa tập `arr` chia được thành các cặp `{x, 2x}` (mỗi cặp một lần đứng theo thứ tự `x` rồi `2x`).

**(b) Phần tử `|x|` nhỏ nhất (`x != 0`) bắt buộc là đầu cặp.** Giả sử trong một cách chia, `x` (với `|x|` nhỏ nhất trong các phần tử chưa xử lý) là phần tử thứ hai của cặp `{x/2, x}`. Khi đó `|x/2| < |x|` và `x/2` là một phần tử chưa xử lý (chưa ghép), mâu thuẫn với `x` có `|x|` nhỏ nhất. Do đó mọi bản của `x` là đầu cặp, và cần đúng `cnt[x]` bản của `2x` (mỗi bản của `x` một bản `2x` riêng). Nếu `cnt[2x] < cnt[x]` thì không chia được. Nếu đủ, ghép `cnt[x]` bản `2x` còn lại `cnt[2x] - cnt[x]` cho bước sau (các bản này không thể ghép với phần tử khác ngoài `4x` hoặc đã bị dùng; việc chọn bản nào cho `x` là tùy ý vì các bản giống hệt nhau). Quy nạp theo số phần tử.

**(c) Số `0`.** `0` chỉ ghép được với `0` nên số bản `0` phải chẵn; các cặp `(0, 0)` độc lập với các phần tử khác (và không ảnh hưởng đến phần còn lại).

**(d) Cách 3.** Với số dương, `|x| = x` nên sắp xếp tăng dần; với số âm, `|x| = -x` tăng ⇔ `x` giảm dần. Mỗi nhóm xử lý theo `|x|` tăng dần như Cách 2; số dương không ghép với số âm (dấu bất biến khi nhân đôi).

**(e) Cách 4.** Cặp `(x, 2x)` giữ nguyên `(dấu, phần lẻ)`, tăng bậc `k` thêm `1`: nối hai bậc liền kề trong một chuỗi. Trong chuỗi, bậc thấp nhất `0` chỉ ghép với bậc `1` (như (b)); phần dư ở bậc `1` phải ghép với bậc `2`... Điều kiện `c_k >= r_{k-1}` và `r_last = 0` là đúng điều kiện chia được.

**Kiểm chứng bằng máy.** Oracle **độc lập**: quay lui thử mọi cách ghép (`n <= 12`); và **dựng được lời giải**: khi một cách trả `True` ta dựng chính xác một **cách xếp lại** `arr[2i+1] = 2 * arr[2i]` và kiểm tra nó là hoán vị của `arr` (bằng chứng cho `True`). Với `False` đối chiếu với quay lui (nhỏ) và với các cách khác (lớn).

In [16]:
def build_reordering(arr: List[int]):
    """Dựng một cách xếp lại hợp lệ bằng tham lam theo |x| (hoặc None). Dùng chính cấu trúc ghép cặp."""
    cnt = Counter(arr)
    out = []
    if cnt[0] % 2:
        return None
    out.extend([0] * cnt[0])
    for x in sorted(cnt, key=abs):
        if x == 0:
            continue
        c = cnt[x]
        if c == 0:
            continue
        if cnt[2 * x] < c:
            return None
        cnt[2 * x] -= c
        cnt[x] = 0
        for _ in range(c):
            out.extend([x, 2 * x])
    return out

def is_valid_reordering(arr: List[int], out) -> bool:
    return (out is not None and len(out) == len(arr) and Counter(out) == Counter(arr)
            and all(out[2 * i + 1] == 2 * out[2 * i] for i in range(len(out) // 2)))

rng = random.Random(1954)
for _ in range(500):
    n = rng.choice([2, 4, 6, 8, 10, 12])
    arr = [rng.randint(-6, 6) for _ in range(n)]
    b = reorder_brute(arr)
    out = build_reordering(arr)
    assert b == (out is not None), arr
    if out is not None:
        assert is_valid_reordering(arr, out), (arr, out)
print("quay lui thử mọi cách ghép và bộ dựng lời giải (kèm kiểm tra hợp lệ) khớp nhau trên 500 mảng nhỏ")

quay lui thử mọi cách ghép và bộ dựng lời giải (kèm kiểm tra hợp lệ) khớp nhau trên 500 mảng nhỏ


## 11. Time / Space Complexity

Gọi `n` là độ dài mảng, `d` là số giá trị khác nhau.

| Cách | Time | Space |
|---|---|---|
| 1. Quay lui | mũ | `O(n)` |
| 2. `Counter` + `|x|` | `O(n + d log d)` | `O(n)` |
| 3. Tách dấu | `O(n + d log d)` | `O(n)` |
| 4. Chuỗi gấp đôi | `O(n + d log V)` | `O(d)` |

Đo thử với `n = 3 * 10^4` (cỡ tối đa):

In [17]:
rng = random.Random(11)
n = 3 * 10 ** 4
base = [rng.randint(-50000, 50000) for _ in range(n // 2)]
arr_true = base + [2 * x for x in base]
rng.shuffle(arr_true)
arr_false = arr_true[:]
arr_false[0] += 1
for label, arr in (("True", arr_true), ("False", arr_false)):
    for name, fn in (("Cách 2", sol_counter), ("Cách 3", sol_split), ("Cách 4", sol_chain), ("nộp", Solution().canReorderDoubled)):
        t0 = time.perf_counter()
        got = fn(list(arr))
        dt = (time.perf_counter() - t0) * 1000
        print(f"{label:5} {name:7} n={n}: {dt:7.2f} ms -> {got}")

True  Cách 2  n=30000:   21.26 ms -> True
True  Cách 3  n=30000:   19.01 ms -> True
True  Cách 4  n=30000:   22.09 ms -> True


True  nộp     n=30000:   19.25 ms -> True
False Cách 2  n=30000:   11.37 ms -> False
False Cách 3  n=30000:   15.15 ms -> False


False Cách 4  n=30000:   16.69 ms -> False


False nộp     n=30000:   11.43 ms -> False


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai (tìm bằng tìm kiếm có seed), đối chiếu mọi cách với oracle, test ngẫu nhiên có seed (đúng ràng buộc) và kiểm tra lời giải dựng lại.

In [18]:
def solutions_list():
    return {
        "counter": reorder_counter,
        "split": reorder_split_sign,
        "chains": reorder_chains,
        "sol_counter": sol_counter,
        "sol_split": sol_split,
        "sol_chain": sol_chain,
        "final": Solution().canReorderDoubled,
    }

def run_all(arr, expected=None, brute_limit=12):
    ref = reorder_brute(arr) if len(arr) <= brute_limit else (build_reordering(arr) is not None)
    out = build_reordering(arr)
    assert (out is not None) == ref
    if out is not None:
        assert is_valid_reordering(arr, out), (arr, out)
    for name, fn in solutions_list().items():
        assert fn(list(arr)) == ref, (name, arr)
    if expected is not None:
        assert ref == expected, ("kỳ vọng sai", arr, expected, ref)
    return ref

# 12.1 ví dụ đề
for arr, expected in examples:
    assert run_all(arr, expected) == expected
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([0, 0], True),                           # cặp (0, 0)
    ([0, 1], False),                          # 0 không ghép với 1
    ([1, 2], True),                           # (1, 2)
    ([2, 1], True),                           # thứ tự đầu vào tùy ý
    ([1, 1], False),                          # 1 != 2 * 1
    ([-1, -2], True),                         # (-1, -2)
    ([-2, -1], True),
    ([1, -2], False),                         # khác dấu
    ([0, 0, 0, 0], True),                     # hai cặp (0, 0)
    ([0, 0, 0, 2], False),                    # số 0 lẻ
    ([1, 2, 4, 8], True),                     # (1, 2), (4, 8)
    ([1, 2, 2, 4], True),                     # (1, 2), (2, 4)
    ([1, 2, 4, 4], False),                    # 4 không còn 8
    ([2, 4, 4, 8], True),                     # (2, 4), (4, 8)
    ([4, -2, 2, -4], True),
    ([-4, -2, 1, 2], True),                   # (-2, -4), (1, 2)
    ([1, 2, 1, 2], True),                     # hai cặp (1, 2)
    ([100000, 50000], True),                  # biên
    ([-100000, -50000], True),
    ([3, 6, 12, 24, 6, 12], True),            # chuỗi 3,6,12,24 với số lần 1,2,2,1: (3,6),(6,12),(12,24)... hai cặp (6,12): ghép được
]
for arr, expected in edge:
    run_all(arr, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 20 ca


In [19]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm có seed rồi xác nhận lời giải đúng không bị lỗi)
def wrong_sort_by_value(arr):                      # sắp xếp theo giá trị thường (không phải |x|)
    cnt = Counter(arr)
    if cnt[0] % 2:
        return False
    for x in sorted(cnt):
        if x == 0:
            continue
        if cnt[2 * x] < cnt[x]:
            return False
        cnt[2 * x] -= cnt[x]
    return True

def wrong_no_subtract(arr):                        # kiểm tra cnt[2x] >= cnt[x] nhưng quên trừ số đã ghép
    cnt = Counter(arr)
    if cnt[0] % 2:
        return False
    for x in sorted(cnt, key=abs):
        if x == 0:
            continue
        if cnt[2 * x] < cnt[x]:
            return False
    return True

def wrong_presence_only(arr):                      # chỉ kiểm tra 2x (hoặc x/2) có mặt, không đếm số lượng
    s = set(arr)
    if arr.count(0) % 2:
        return False
    return all((2 * x in s) or (x % 2 == 0 and x // 2 in s) for x in s if x != 0)

def wrong_insertion_order(arr):                    # duyệt Counter theo thứ tự chèn (không sắp xếp)
    cnt = Counter(arr)
    if cnt[0] % 2:
        return False
    for x in list(cnt):
        if x == 0:
            continue
        if cnt[2 * x] < cnt[x]:
            return False
        cnt[2 * x] -= cnt[x]
    return True

def find_counter(wrong, tries=30000, seed=5):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.choice([2, 4, 6, 8])
        arr = [rng.randint(-8, 8) for _ in range(n)]
        if wrong(list(arr)) != reorder_brute(arr):
            return arr
    return None

counters = {}
for name, fn in (("sắp xếp theo giá trị", wrong_sort_by_value), ("quên trừ số đã ghép", wrong_no_subtract),
                 ("chỉ kiểm tra có mặt", wrong_presence_only), ("duyệt theo thứ tự chèn", wrong_insertion_order)):
    arr = find_counter(fn)
    assert arr is not None, name
    counters[name] = arr
    print(f"{name:26} arr={arr}: sai -> {fn(list(arr))}, đúng -> {reorder_brute(arr)}")
    run_all(arr)
print("12.3 phản ví dụ: OK")

sắp xếp theo giá trị       arr=[-8, -4]: sai -> False, đúng -> True
quên trừ số đã ghép        arr=[2, 1]: sai -> False, đúng -> True
chỉ kiểm tra có mặt        arr=[2, 4, 4, 2, -1, -2, -4, -4]: sai -> True, đúng -> False
duyệt theo thứ tự chèn     arr=[2, 1]: sai -> False, đúng -> True
12.3 phản ví dụ: OK


In [20]:
# 12.4 ngẫu nhiên nhỏ: mọi cách so với quay lui (n <= 12), kèm kiểm tra lời giải dựng lại
rng = random.Random(4954)
trues = 0
for _ in range(2000):
    n = rng.choice([2, 4, 6, 8, 10, 12])
    mode = rng.random()
    if mode < 0.5:
        base = [rng.randint(-6, 6) for _ in range(n // 2)]
        arr = base + [2 * x for x in base]               # chắc chắn True
        rng.shuffle(arr)
    else:
        arr = [rng.randint(-8, 8) for _ in range(n)]
    trues += run_all(arr)
print(f"12.4 ngẫu nhiên nhỏ (2000 ca): OK, {trues} ca True")

# 12.5 trung bình: n tới 400 (không quay lui): các cách khớp nhau và True khi dựng được
rng = random.Random(5954)
for _ in range(300):
    n = rng.choice(range(14, 401, 2))
    if rng.random() < 0.5:
        base = [rng.randint(-1000, 1000) for _ in range(n // 2)]
        arr = base + [2 * x for x in base]
        rng.shuffle(arr)
        if rng.random() < 0.3:
            arr[rng.randrange(n)] += rng.choice([-1, 1])
    else:
        arr = [rng.randint(-50, 50) for _ in range(n)]
    run_all(arr, brute_limit=0)
print("12.5 ngẫu nhiên n tới 400 (300 ca): OK")

# 12.6 lớn n = 3 * 10^4 (đúng ràng buộc): True / False nhiều dạng
rng = random.Random(6954)
n = 3 * 10 ** 4
shapes = {}
base = [rng.randint(-50000, 50000) for _ in range(n // 2)]
shapes["True ngẫu nhiên"] = (base + [2 * x for x in base], True)
shapes["chuỗi gấp đôi dài"] = ([1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768] * 1, True)
shapes["toàn 0"] = ([0] * n, True)
shapes["toàn 1 (False)"] = ([1] * n, False)
shapes["một nửa 1 một nửa 2"] = ([1] * (n // 2) + [2] * (n // 2), True)
shapes["âm: nửa -1 nửa -2"] = ([-1] * (n // 2) + [-2] * (n // 2), True)
shapes["nhiều số 0 và một cặp (1, 2)"] = ([0] * (n - 2) + [1, 2], True)
shapes["số 0 lẻ (False)"] = ([0] * (n - 1) + [3], False)
for name, (arr, expected) in shapes.items():
    arr = list(arr)
    if len(arr) % 2:
        continue
    random.Random(1).shuffle(arr)
    got = {fname: fn(list(arr)) for fname, fn in solutions_list().items()}
    assert len(set(got.values())) == 1, (name, got)
    assert got["final"] == expected, (name, got["final"], expected)
    print(f"  {name:22} n={len(arr)} -> {got['final']}")
print("12.6 n = 3 * 10^4: OK")

# 12.7 tính chất: bất biến khi hoán vị đầu vào; nhân mọi phần tử với c != 0 giữ đáp án; đổi dấu mọi phần tử giữ đáp án
rng = random.Random(7954)
for _ in range(300):
    n = rng.choice([2, 4, 6, 8, 10])
    arr = [rng.randint(-8, 8) for _ in range(n)]
    base = Solution().canReorderDoubled(list(arr))
    sh = arr[:]
    rng.shuffle(sh)
    assert Solution().canReorderDoubled(sh) == base
    c = rng.choice([-3, -1, 2, 5])
    assert Solution().canReorderDoubled([c * x for x in arr]) == base
print("12.7 tính chất (hoán vị, nhân hằng số khác 0): OK")

12.4 ngẫu nhiên nhỏ (2000 ca): OK, 1030 ca True


12.5 ngẫu nhiên n tới 400 (300 ca): OK


  True ngẫu nhiên        n=30000 -> True
  chuỗi gấp đôi dài      n=16 -> True
  toàn 0                 n=30000 -> True
  toàn 1 (False)         n=30000 -> False
  một nửa 1 một nửa 2    n=30000 -> True


  âm: nửa -1 nửa -2      n=30000 -> True
  nhiều số 0 và một cặp (1, 2) n=30000 -> True
  số 0 lẻ (False)        n=30000 -> False
12.6 n = 3 * 10^4: OK
12.7 tính chất (hoán vị, nhân hằng số khác 0): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Sắp xếp theo giá trị thay vì `|x|`

Với số âm, `x` nhỏ nhất (rất âm) lại là giá trị **lớn nhất** về `|x|`; xử lý nó đầu tiên sẽ tìm `2x` (còn âm hơn) và sai.

In [21]:
arr = counters["sắp xếp theo giá trị"]
print("arr =", arr, "| sai ->", wrong_sort_by_value(list(arr)), "| đúng ->", Solution().canReorderDoubled(list(arr)))
assert wrong_sort_by_value(list(arr)) != Solution().canReorderDoubled(list(arr))
print("ví dụ 3: [4, -2, 2, -4] ->", wrong_sort_by_value([4, -2, 2, -4]), "(sai) so với", Solution().canReorderDoubled([4, -2, 2, -4]))

arr = [-8, -4] | sai -> False | đúng -> True
ví dụ 3: [4, -2, 2, -4] -> False (sai) so với True


Vì sao sai: `-4` đứng đầu và cần `-8`, nhưng đúng ra `-4 = 2 * (-2)` là phần tử **thứ hai** của cặp. **Đúng:** `sorted(cnt, key=abs)`.

### Lỗi 2 — Quên trừ số bản đã ghép

Kiểm tra `cnt[2x] >= cnt[x]` nhưng **không** trừ `cnt[x]` khỏi `cnt[2x]`: các bản `2x` đã dùng làm "cuối cặp" lại được dùng tiếp làm "đầu cặp" của `4x`.

In [22]:
arr = counters["quên trừ số đã ghép"]
print("arr =", arr, "| sai ->", wrong_no_subtract(list(arr)), "| đúng ->", Solution().canReorderDoubled(list(arr)))
assert wrong_no_subtract(list(arr)) != Solution().canReorderDoubled(list(arr))

arr = [2, 1] | sai -> False | đúng -> True


Vì sao sai: mỗi bản `2x` chỉ thuộc đúng một cặp; nếu không trừ thì cùng một bản `2x` được tính hai lần (làm cuối cặp của `x` và đầu cặp của `4x`). **Đúng:** `cnt[2 * x] -= cnt[x]`.

### Lỗi 3 — Chỉ kiểm tra `2x` có mặt (không đếm số lượng)

Mỗi `2x` chỉ ghép với **một** `x`; cần so số lượng.

In [23]:
arr = counters["chỉ kiểm tra có mặt"]
print("arr =", arr, "| sai ->", wrong_presence_only(list(arr)), "| đúng ->", Solution().canReorderDoubled(list(arr)))
assert wrong_presence_only(list(arr)) != Solution().canReorderDoubled(list(arr))

arr = [2, 4, 4, 2, -1, -2, -4, -4] | sai -> True | đúng -> False


Vì sao sai: ví dụ `[1, 1, 2, 4]`: `1` cần hai `2` nhưng chỉ có một. **Đúng:** `cnt[2x] >= cnt[x]` rồi trừ.

### Lỗi 4 — Duyệt `Counter` theo thứ tự chèn

Thứ tự khóa của `Counter` là thứ tự xuất hiện trong mảng, không phải `|x|` tăng dần; tham lam cần đúng thứ tự.

In [24]:
arr = counters["duyệt theo thứ tự chèn"]
print("arr =", arr, "| sai ->", wrong_insertion_order(list(arr)), "| đúng ->", Solution().canReorderDoubled(list(arr)))
assert wrong_insertion_order(list(arr)) != Solution().canReorderDoubled(list(arr))

arr = [2, 1] | sai -> False | đúng -> True


Vì sao sai: nếu `2x` đứng trước `x` trong mảng, ta xử lý `2x` (tìm `4x`) trước khi `x` lấy đi `2x` của nó. **Đúng:** sắp xếp theo `|x|`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "ghép phần tử với bội/gấp đôi của nó" ⇒ **tham lam theo thứ tự `|x|` tăng dần** với bộ đếm.
- **Lý do:** phần tử nhỏ nhất theo `|x|` không thể là phần tử thứ hai của cặp.
- **Số âm** và **số `0`** là hai bẫy: dùng `key=abs`; `0` chỉ ghép với `0` (kiểm tra `cnt[0]` chẵn).
- Tổng quát hóa: `arr[2i+1] = k * arr[2i]` (cùng kỹ thuật với `k` bất kỳ).
- Cùng họ: Check If N and Its Double Exist (1346), Divide Array in Sets of K Consecutive Numbers (1296), Hand of Straights (846), Largest Divisible Subset (368).
- Test nên có: `[0, 0]`, `[0, 1]`, số âm, chuỗi gấp đôi dài, số lượng không cân bằng, `n = 3 * 10^4`.

## 15. Biến thể

1. **Trả về một cách xếp lại** hợp lệ (bài tập 5).
2. **Bội `k` bất kỳ:** `arr[2i+1] = k * arr[2i]` (`k >= 2`): thay `2x` bằng `k * x`.
3. **Số nhóm tối thiểu** nếu cho phép bỏ phần tử: khó hơn.
4. Bài liên quan: 1346, 1296.

Dưới đây là biến thể 2 (bội `k`) kiểm bằng quay lui.

In [25]:
def reorder_k_multiples(arr: List[int], k: int) -> bool:
    cnt = Counter(arr)
    if cnt[0] % 2:
        return False
    for x in sorted(cnt, key=abs):
        if x == 0:
            continue
        if cnt[k * x] < cnt[x]:
            return False
        cnt[k * x] -= cnt[x]
    return True

def reorder_k_brute(arr: List[int], k: int) -> bool:
    def go(rest):
        if not rest:
            return True
        e, others = rest[0], rest[1:]
        for idx, y in enumerate(others):
            if y == k * e or (e % k == 0 and y == e // k):
                if go(others[:idx] + others[idx + 1:]):
                    return True
        return False
    return go(list(arr))

rng = random.Random(15)
for _ in range(400):
    k = rng.choice([2, 3, 4])
    arr = [rng.randint(-9, 9) for _ in range(rng.choice([2, 4, 6, 8]))]
    assert reorder_k_multiples(arr, k) == reorder_k_brute(arr, k), (arr, k)
print("biến thể bội k khớp quay lui; ví dụ:", reorder_k_multiples([1, 3, 9, 3], 3), reorder_k_multiples([1, 3, 3, 9, 3, 3], 3))

biến thể bội k khớp quay lui; ví dụ: True False


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quay lui | Định nghĩa | mũ | `O(n)` | Oracle, `n <= 12` |
| 2. `Counter` + `|x|` | Tham lam | `O(n + d log d)` | `O(n)` | **Cách nộp chính** |
| 3. Tách dấu | Dương tăng, âm giảm | `O(n + d log d)` | `O(n)` | Không dùng `key=abs` |
| 4. Chuỗi gấp đôi | Nhóm theo phần lẻ | `O(n + d log V)` | `O(d)` | Hiểu cấu trúc bài toán |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng, rồi chạy để kiểm tra:

```python
cnt = Counter([4, -2, 2, -4])
print(sorted(cnt, key=abs))
print(cnt[8], cnt[2])
print(sorted([4, -2, 2, -4]))
```

In [26]:
# Bài 1: ghi dự đoán của bạn
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `arr = [1, 2, 2, 4, 4, 8, 8, 16]`: trả lời `True/False` và liệt kê các cặp `(x, 2x)` (một cách chia).

In [27]:
# Bài 2: điền kết quả tính tay
hand_answer = None
hand_pairs = None             # ví dụ [(1, 2), (2, 4), ...]

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành cách `Counter` + `|x|`.

In [28]:
def exercise_3_counter(arr: List[int]) -> bool:
    cnt = Counter(arr)
    if cnt[0] % 2:
        return False
    for x in sorted(cnt, key=abs):
        if x == 0:
            continue
        # TODO 1: nếu cnt[2 * x] < cnt[x] thì trả False
        # TODO 2: trừ cnt[x] khỏi cnt[2 * x]
        pass
    return True


# for arr, expected in examples:
#     assert exercise_3_counter(arr) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, hãy tự viết `exercise_4_chains(arr)` (phân rã theo chuỗi gấp đôi).

In [29]:
def exercise_4_chains(arr: List[int]) -> bool:
    # TODO
    pass


# for arr, expected in examples:
#     assert exercise_4_chains(arr) == expected

### Bài 5 — Mini challenge: trả về một cách xếp lại (khó)

Viết `exercise_5_reorder(arr)` trả về một danh sách `out` là **hoán vị** của `arr` thỏa `out[2i + 1] == 2 * out[2i]`, hoặc `None` nếu không có. Có nhiều đáp án; ta kiểm tra bằng hàm kiểm tra hợp lệ.

```python
exercise_5_reorder([4, -2, 2, -4])  -> ví dụ [-2, -4, 2, 4]
exercise_5_reorder([3, 1, 3, 6])    -> None
```

In [30]:
def exercise_5_reorder(arr: List[int]):
    # TODO
    pass


# out = exercise_5_reorder([4, -2, 2, -4])
# assert is_valid_reordering([4, -2, 2, -4], out)
# assert exercise_5_reorder([3, 1, 3, 6]) is None

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `sorted(cnt, key=abs)` sắp theo giá trị tuyệt đối; cùng `|x|` giữ thứ tự chèn.
- Mức 2: khóa chèn theo thứ tự `4, -2, 2, -4`; `|x|` là `4, 2, 2, 4`.
- Mức 3: `[-2, 2, 4, -4]`, `0 1`, `[-4, -2, 2, 4]`.

**Bài 2**
- Mức 1: đếm `1:1, 2:2, 4:2, 8:2, 16:1`.
- Mức 2: `1` ghép với một `2`; hai `2`... một còn lại ghép với `4`, v.v.
- Mức 3: `True`: `(1, 2), (2, 4), (4, 8), (8, 16)`.

**Bài 3**
- Mức 1: `if cnt[2 * x] < cnt[x]: return False`.
- Mức 2: `cnt[2 * x] -= cnt[x]`.
- Mức 3: vòng lặp theo `key=abs` và bỏ qua `x = 0`.

**Bài 4**
- Mức 1: tách `x = dấu * 2^k * lẻ`.
- Mức 2: gom số lần theo `(dấu, lẻ)` và bậc `k`.
- Mức 3: `r = 0`; với mỗi bậc: `c < r` thì `False`; `r = c - r`; cuối `r == 0`.

**Bài 5**
- Mức 1: kiểm tra `cnt[0]` chẵn; thêm các cặp `(0, 0)`.
- Mức 2: với mỗi `x` theo `|x|`: thêm `cnt[x]` cặp `(x, 2x)`; trừ `cnt[2x]`.
- Mức 3: nếu một bước thiếu `2x` thì trả `None`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [31]:
cnt = Counter([4, -2, 2, -4])
print(sorted(cnt, key=abs))
print(cnt[8], cnt[2])
print(sorted([4, -2, 2, -4]))

assert sorted(cnt, key=abs) == [-2, 2, 4, -4]
assert (cnt[8], cnt[2]) == (0, 1)
assert sorted([4, -2, 2, -4]) == [-4, -2, 2, 4]

[-2, 2, 4, -4]
0 1
[-4, -2, 2, 4]


### Đáp án Bài 2

`True`: các cặp `(1, 2), (2, 4), (4, 8), (8, 16)`.

In [32]:
arr = [1, 2, 2, 4, 4, 8, 8, 16]
hand_answer_solution = True
hand_pairs_solution = [(1, 2), (2, 4), (4, 8), (8, 16)]
assert Solution().canReorderDoubled(list(arr)) is hand_answer_solution and reorder_brute(arr)
assert all(b == 2 * a for a, b in hand_pairs_solution)
assert Counter(x for p in hand_pairs_solution for x in p) == Counter(arr)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [33]:
def exercise_3_counter_solution(arr: List[int]) -> bool:
    cnt = Counter(arr)
    if cnt[0] % 2:
        return False
    for x in sorted(cnt, key=abs):
        if x == 0:
            continue
        if cnt[2 * x] < cnt[x]:
            return False
        cnt[2 * x] -= cnt[x]
    return True

for arr, expected in examples:
    assert exercise_3_counter_solution(arr) == expected
rng = random.Random(3)
for _ in range(300):
    arr = [rng.randint(-6, 6) for _ in range(rng.choice([2, 4, 6, 8]))]
    assert exercise_3_counter_solution(arr) == reorder_brute(arr)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [34]:
def exercise_4_chains_solution(arr: List[int]) -> bool:
    cnt = Counter(arr)
    if cnt[0] % 2:
        return False
    chains = {}
    for x, c in cnt.items():
        if x == 0:
            continue
        sign = 1 if x > 0 else -1
        v, k = abs(x), 0
        while v % 2 == 0:
            v //= 2
            k += 1
        chains.setdefault((sign, v), {})[k] = c
    for chain in chains.values():
        r = 0
        for k in range(max(chain) + 1):
            c = chain.get(k, 0)
            if c < r:
                return False
            r = c - r
        if r:
            return False
    return True

for arr, expected in examples:
    assert exercise_4_chains_solution(arr) == expected
rng = random.Random(4)
for _ in range(300):
    arr = [rng.randint(-6, 6) for _ in range(rng.choice([2, 4, 6, 8]))]
    assert exercise_4_chains_solution(arr) == reorder_brute(arr)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [35]:
def exercise_5_reorder_solution(arr: List[int]):
    cnt = Counter(arr)
    if cnt[0] % 2:
        return None
    out = [0] * cnt[0]
    for x in sorted(cnt, key=abs):
        if x == 0:
            continue
        c = cnt[x]
        if c == 0:
            continue
        if cnt[2 * x] < c:
            return None
        cnt[2 * x] -= c
        cnt[x] = 0
        for _ in range(c):
            out.extend([x, 2 * x])
    return out

out = exercise_5_reorder_solution([4, -2, 2, -4])
assert is_valid_reordering([4, -2, 2, -4], out)
assert exercise_5_reorder_solution([3, 1, 3, 6]) is None
rng = random.Random(5)
for _ in range(300):
    arr = [rng.randint(-6, 6) for _ in range(rng.choice([2, 4, 6, 8]))]
    o = exercise_5_reorder_solution(arr)
    assert (o is not None) == reorder_brute(arr)
    if o is not None:
        assert is_valid_reordering(arr, o)
print("Bài 5 OK", out)

Bài 5 OK [-2, -4, 2, 4]


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (tham lam theo `|x|`)

```python
class Solution:
    def canReorderDoubled(self, arr: List[int]) -> bool:
        cnt = Counter(arr)
        if cnt[0] % 2:                        # số 0 chỉ ghép với 0
            return False
        for x in sorted(cnt, key=abs):        # |x| tăng dần: phần tử nhỏ nhất bắt buộc là đầu cặp
            if x == 0:
                continue
            if cnt[2 * x] < cnt[x]:           # cần đủ cnt[x] bản của 2x
                return False
            cnt[2 * x] -= cnt[x]
        return True
```

Công thức nhớ nhanh: duyệt theo `|x|` tăng dần; `cnt[2x] -= cnt[x]` (fail nếu thiếu); `cnt[0]` chẵn.

### Checklist trước khi submit

- [ ] Sắp xếp theo `|x|` (`key=abs`), không theo giá trị?
- [ ] Số `0` chỉ ghép với `0` (số lượng chẵn, bỏ qua `x = 0` trong vòng lặp)?
- [ ] Trừ `cnt[2x] -= cnt[x]` sau khi ghép?
- [ ] So **số lượng** `cnt[2x] >= cnt[x]` rồi trừ (không chỉ kiểm tra có mặt)?
- [ ] Không duyệt `Counter` theo thứ tự chèn?
- [ ] Đã thử: `[0, 0]`, `[0, 1]`, số âm, chuỗi gấp đôi dài, `n = 3 * 10^4`?
- [ ] Tên method đúng `canReorderDoubled(self, arr)` và trả `bool`?